In [1]:
# -*- coding: utf-8 -*-
"""
=============================================================================
S_PricingError_Daily —— 定价误差因子 (ω4 定价公理 | 全新残差维度)
=============================================================================
信号逻辑：收盘价偏离日内VWAP的程度 → Z-score → 截面rank
VWAP = 市场共识价格，偏离越大 = 修正预期越强
=============================================================================
"""

import dai
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


def main(datasources, start_date, end_date):
    bar1m = datasources["bar1m"]
    BUFFER_DAYS = 120
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=BUFFER_DAYS)).strftime('%Y-%m-%d %H:%M:%S')

    sql = f"""
    WITH minute_data AS (
        SELECT
            date_trunc('day', date)::DATE AS trade_date,
            date AS dt,
            instrument,
            close,
            volume,
            amount
        FROM {bar1m}
        WHERE close > 0 AND volume > 0
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY instrument, trade_date ORDER BY dt DESC
        ) AS rn
        FROM minute_data
    ),
    last_close AS (
        SELECT trade_date, instrument, close AS close_price
        FROM ranked WHERE rn = 1
    ),
    daily_agg AS (
        SELECT
            trade_date, instrument,
            SUM(amount) / NULLIF(SUM(volume), 0) AS vwap,
            SUM(volume) AS total_volume,
            MAX(close) AS day_high,
            MIN(close) AS day_low,
            COUNT(*) AS n_minutes
        FROM minute_data
        GROUP BY trade_date, instrument
        HAVING COUNT(*) >= 120 AND SUM(volume) > 0
    )
    SELECT
        d.trade_date AS date, d.instrument,
        d.vwap, l.close_price,
        d.total_volume, d.day_high, d.day_low, d.n_minutes
    FROM daily_agg d
    JOIN last_close l
      ON d.trade_date = l.trade_date AND d.instrument = l.instrument
    WHERE d.vwap > 0 AND l.close_price > 0
    ORDER BY d.instrument, d.trade_date
    """

    df = dai.query(sql, filters={'date': [query_start, end_date]}).df()
    df['date'] = pd.to_datetime(df['date'])
    print(f"  SQL: {len(df)} 行, {df['instrument'].nunique()} 只股票")

    eps = 1e-10
    for col in ['vwap', 'close_price', 'total_volume', 'day_high', 'day_low']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 定价偏差: (close - vwap) / vwap, 正值=高估, 负值=低估
    df['pricing_bias'] = (df['close_price'] - df['vwap']) / df['vwap'].clip(lower=eps)

    df = df.sort_values(['instrument', 'date']).reset_index(drop=True)
    df['bias_smooth'] = df.groupby('instrument')['pricing_bias'].transform(
        lambda x: x.rolling(10, min_periods=3).mean()
    )

    def rolling_zscore(group):
        group = group.copy()
        m = group['bias_smooth'].rolling(60, min_periods=20).mean()
        s = group['bias_smooth'].rolling(60, min_periods=20).std()
        group['bias_zscore'] = (group['bias_smooth'] - m) / s.clip(lower=eps)
        return group

    df = df.groupby('instrument', group_keys=False).apply(rolling_zscore)

    # 负偏差(低估) → 高分 → 预期上涨修正
    df['factor'] = df.groupby('date')['bias_zscore'].transform(
        lambda x: (-x).rank(pct=True)
    )

    df = df[(df['date'] >= pd.to_datetime(start_date)) & (df['date'] <= pd.to_datetime(end_date))]
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]}
    ).df()
    stk_pool['date'] = pd.to_datetime(stk_pool['date'])

    result = pd.merge(stk_pool, df[['date', 'instrument', 'factor']], on=['date', 'instrument'], how='left')
    result['factor'] = result['factor'].fillna(0.5).replace([np.inf, -np.inf], 0.5).clip(0.0, 1.0)
    result = result[['date', 'instrument', 'factor']]

    print(f"  最终: {len(result)} 行, mean={result['factor'].mean():.4f}, range=[{result['factor'].min():.4f}, {result['factor'].max():.4f}]")
    return result


if __name__ == '__main__':
    from bigmodule import M
    import structlog
    logger = structlog.get_logger()
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m_selftest'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-10-31 23:59:59'
    logger.info(f"S_PricingError 自检: {start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)
    first_day_na = factor_data[factor_data['date'] == factor_data['date'].min()]['factor'].isna().mean()
    logger.info(f"首日NA率: {first_day_na:.4f}")
    print(f"行数: {len(factor_data)}, 范围: [{factor_data['factor'].min():.4f}, {factor_data['factor'].max():.4f}]")

[2026-07-31 08:56:28] [info     ] S_PricingError 自检: 2024-01-01 00:00:00 ~ 2024-10-31 23:59:59
  SQL: 562119 行, 2203 只股票
  最终: 199000 行, mean=0.5037, range=[0.0005, 1.0000]
[2026-07-31 08:59:41] [info     ] 首日NA率: 0.0000
行数: 199000, 范围: [0.0005, 1.0000]
